# Notebook 1: Transcript Preprocessing\n\nParse raw transcripts and produce training-ready examples in the special-token\nformat defined in `FT_plan.md`. Each training example targets one substantive\nagent turn.\n\n**Input:** `data/raw_transcripts.jsonl` (from notebook 0 or real data drop-in)\n**Output:** `data/train_examples.jsonl`

In [ ]:
import json
import re
import random
from pathlib import Path
from transformers import AutoTokenizer

# ============================================================
# Configuration
# ============================================================
MODEL_DIR = "/path/to/gpt-oss-120b"   # <-- FILL IN: local checkpoint directory
INPUT_FILE = Path("data/raw_transcripts.jsonl")
OUTPUT_FILE = Path("data/train_examples.jsonl")

MIN_AGENT_TOKENS = 15          # skip agent backchannels shorter than this
CONTEXT_POPULATE_RATE = 0.15   # 15% of examples get a populated context block
WINDOW_SIZE_K = 20             # max turns in conversation history window

## Load Tokenizer (for token counting only — no model loaded)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")

## Define System Prompt and Context Templates

In [ ]:
# Fixed system prompt — same for every training example (per FT_plan.md §6.3)
SYSTEM_PROMPT = (
    "You are an outbound sales agent for American Express.\n"
    "Your goal is to identify customer needs and guide the conversation\n"
    "toward [product/outcome]. Use information provided in the context\n"
    "block when available to reference relevant offers, campaigns,\n"
    "or product details. If no context is provided, rely on the\n"
    "conversation history to guide your response."
)

# Metadata templates for the 10-20% of examples with populated context blocks
# In production, these would come from transcript metadata. For now, use templates.
CONTEXT_TEMPLATES = [
    "Product: Corporate Platinum Card\nCampaign: Q2 2025 Spend Bonus",
    "Product: Business Gold Card\nCampaign: New Business Welcome Offer",
    "Product: Corporate Green Card\nCampaign: Annual Fee Waiver Promo",
    "Product: Business Platinum Card\nOffer: 150K MR Points Sign-Up Bonus",
    "Product: Blue Business Plus Card\nCampaign: 0% Intro APR for 12 Months",
]

## Parse Raw Transcripts

In [ ]:
TURN_PATTERN = re.compile(r'^(agent|customer)\(u(\d+)\):\s*(.+)$', re.MULTILINE)


def parse_transcript(raw_text: str) -> list[dict]:
    """Parse raw transcript text into a list of turn dicts."""
    matches = TURN_PATTERN.findall(raw_text)
    turns = []
    for speaker, idx, text in matches:
        turns.append({
            "speaker": speaker,
            "index": int(idx),
            "text": text.strip(),
        })
    return turns


# Load all conversations, keep only converted calls (label=1)
conversations = []
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("label") != 1:
            continue
        turns = parse_transcript(obj["transcript"])
        if len(turns) >= 4:
            conversations.append({
                "conversation_id": obj["conversation_id"],
                "turns": turns,
            })

print(f"Loaded {len(conversations)} converted conversations")
for conv in conversations[:3]:
    print(f"  {conv['conversation_id']}: {len(conv['turns'])} turns")

## Helper Functions: Filtering and Windowing

In [ ]:
def is_substantive_agent_turn(text: str) -> bool:
    """Returns True if agent turn has >= MIN_AGENT_TOKENS tokens.
    Trivial backchannels ('mhm', 'right', 'I see') are filtered out as
    prediction targets but kept in conversation history as context.
    """
    token_count = len(tokenizer.encode(text, add_special_tokens=False))
    return token_count >= MIN_AGENT_TOKENS


def window_turns(turns: list[dict], target_idx: int) -> list[dict]:
    """Apply conversation history windowing (per FT_plan.md §6.2).

    If the history before the target turn is <= WINDOW_SIZE_K turns, keep all.
    Otherwise keep first 2 turns (opening framing) + last (K-2) turns (recent context).
    """
    history = turns[:target_idx]
    if len(history) <= WINDOW_SIZE_K:
        return history
    first_part = history[:2]
    last_part = history[-(WINDOW_SIZE_K - 2):]
    return first_part + last_part

## Build Training Examples\n\nFor each substantive agent turn, construct:\n- `prefix`: system prompt + context block + conversation history + `<|agent|>` marker\n- `target`: the agent's actual utterance (loss is computed only on this)\n\nStoring `prefix` and `target` separately makes loss mask construction in notebook 2 trivial.

In [ ]:
def build_example(turns: list[dict], target_turn_pos: int, conv_id: str) -> dict:
    """Build one training example for the agent turn at target_turn_pos."""

    # Window the conversation history (everything before the target turn)
    history_turns = window_turns(turns, target_turn_pos)
    target_text = turns[target_turn_pos]["text"]

    # Randomly decide whether to populate the context block
    if random.random() < CONTEXT_POPULATE_RATE:
        context_content = random.choice(CONTEXT_TEMPLATES)
    else:
        context_content = ""

    # Build conversation history string with special-token markers
    conv_lines = []
    for t in history_turns:
        marker = "<|agent|>" if t["speaker"] == "agent" else "<|customer|>"
        conv_lines.append(f"{marker}{t['text']}")
    conversation_str = "\n".join(conv_lines)

    # Assemble the prefix (everything the model sees as input — loss masked)
    prefix = (
        f"<|system|>{SYSTEM_PROMPT}<|/system|>\n"
        f"<|context|>{context_content}<|/context|>\n"
        f"<|conversation|>\n"
        f"{conversation_str}\n"
        f"<|/conversation|>\n"
        f"<|agent|>"
    )

    return {
        "conversation_id": conv_id,
        "target_turn_index": turns[target_turn_pos]["index"],
        "prefix": prefix,
        "target": target_text,
        "full_text": prefix + target_text,
    }

## Generate All Training Examples

In [ ]:
all_examples = []
skipped_trivial = 0

for conv in conversations:
    for i, turn in enumerate(conv["turns"]):
        if turn["speaker"] != "agent":
            continue
        if not is_substantive_agent_turn(turn["text"]):
            skipped_trivial += 1
            continue
        example = build_example(conv["turns"], i, conv["conversation_id"])
        all_examples.append(example)

print(f"Generated {len(all_examples)} training examples from {len(conversations)} conversations")
print(f"Skipped {skipped_trivial} trivial agent turns (< {MIN_AGENT_TOKENS} tokens)")

## Validation & Statistics

In [ ]:
# Token length distributions (approximate — uses base tokenizer without special tokens)
full_lengths = [len(tokenizer.encode(ex["full_text"])) for ex in all_examples]
target_lengths = [len(tokenizer.encode(ex["target"])) for ex in all_examples]
prefix_lengths = [len(tokenizer.encode(ex["prefix"])) for ex in all_examples]

n_with_context = sum(
    1 for ex in all_examples if "<|context|><|/context|>" not in ex["prefix"]
)

print(f"Total training examples: {len(all_examples)}")
print(f"Examples with populated context: {n_with_context} "
      f"({100 * n_with_context / len(all_examples):.1f}%)")
print()
print(f"Full sequence token lengths:")
print(f"  min={min(full_lengths)}, max={max(full_lengths)}, "
      f"mean={sum(full_lengths)/len(full_lengths):.0f}")
print(f"Prefix (masked) token lengths:")
print(f"  min={min(prefix_lengths)}, max={max(prefix_lengths)}, "
      f"mean={sum(prefix_lengths)/len(prefix_lengths):.0f}")
print(f"Target (loss) token lengths:")
print(f"  min={min(target_lengths)}, max={max(target_lengths)}, "
      f"mean={sum(target_lengths)/len(target_lengths):.0f}")

# Flag long sequences
MAX_CTX = 4096  # adjust to model's actual context length
n_over = sum(1 for l in full_lengths if l > MAX_CTX)
if n_over > 0:
    print(f"\n⚠ {n_over} examples exceed {MAX_CTX} tokens — will be truncated in notebook 2")
else:
    print(f"\nAll examples fit within {MAX_CTX} token context window.")

## Preview a Training Example

In [ ]:
# Show one example to visually verify the format
ex = all_examples[0]
print(f"Conversation: {ex['conversation_id']}, target turn: u{ex['target_turn_index']}")
print()
print("=== PREFIX (will be loss-masked) ===")
print(ex["prefix"][:600])
print("...")
print()
print("=== TARGET (loss computed here) ===")
print(ex["target"])

## Save to JSONL

In [ ]:
OUTPUT_FILE.parent.mkdir(exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for ex in all_examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Saved {len(all_examples)} training examples to {OUTPUT_FILE}")